# **ColabSeqDisplay · Predict**

<img src="https://img.shields.io/badge/Paper-not%20yet%20posted-lightgrey" style="max-width: 100%;">
<a href="https://github.com/JasonJiangs/ColabSeqDisplay"><img src="https://img.shields.io/badge/Github-black?logo=github" style="max-width: 100%;"></a>
<a href="https://colab.research.google.com/github/JasonJiangs/ColabSeqDisplay/blob/main/colab/ColabSeqDisplay_Predict.ipynb"><img src="https://img.shields.io/badge/Open%20in-Colab-F9AB00?logo=googlecolab&logoColor=white" style="max-width: 100%;"></a>
<a href="https://github.com/JasonJiangs/ColabSeqDisplay"><img src="https://img.shields.io/badge/License-see%20repository-lightgrey" style="max-width: 100%;"></a>

- **You have a `model_bundle.zip` and a list of variants you have not made yet.** This notebook scores them and hands back a ranked table. That is all it does.

- The bundle carries everything the model needs — the backbone name, the LoRA weights, the head, the library spec and the frozen pooling positions — so the variants you type here are built against the same wild type, at the same positions, as the ones it was trained on. A bundle from a different library is refused rather than quietly mis-scored.

- Nothing is uploaded anywhere. The bundle is a file on your computer, and the predictions come back as a CSV you download.

- **Three notebooks.** [ColabSeqDisplay](https://colab.research.google.com/github/JasonJiangs/ColabSeqDisplay/blob/main/colab/ColabSeqDisplay.ipynb) trains, evaluates and exports a model. [ColabSeqDisplay_Prepare](https://colab.research.google.com/github/JasonJiangs/ColabSeqDisplay/blob/main/colab/ColabSeqDisplay_Prepare.ipynb) is run once per new protein and makes the two optional input files. [ColabSeqDisplay_Predict](https://colab.research.google.com/github/JasonJiangs/ColabSeqDisplay/blob/main/colab/ColabSeqDisplay_Predict.ipynb) scores new variants with a model you already trained.

<font color="red">⚠️ <b>Before you run anything:</b> the run-button cell installs this package from https://github.com/JasonJiangs/ColabSeqDisplay.git. If your runtime cannot reach that address, the cell stops there and quotes what `git` or `pip` said — put a fork's URL, or the path of a folder you uploaded to this runtime, in the field at the top of that cell.</font>

# How to start

## 1 · Switch this runtime to a GPU

`Runtime` ▸ `Change runtime type` ▸ **T4 GPU** ▸ `Save`. Colab restarts the runtime, which takes a few seconds and clears anything you had already run.

## 2 · Click the run-button

Hover over the cell below and click ▶ on its left. The cell installs the package (1–2 minutes the first time) and then draws the panel that **is** this notebook: every choice you make is a field or a button in it. There is no code to write and no other cell to edit.

## 3 · Which GPU

- **T4 — <font color="red">free</font>.** 16 GB, which is enough for most of this. <font color="red">It is also unstable: free sessions are pre-empted, drop their connection, and are capped in length, so a run measured in hours is a run you will probably lose halfway.</font> Mount Drive first — see below — and a dropped session costs you the run in progress rather than everything you have done.
- **L4 (needs Colab Pro).** 24 GB — the smallest card that runs the three backbones the registry says will not fit a T4 (`SaProt-1.3B`, `ProtT5-XL`, `Ankh-large`) at all, and enough session stability to finish a long run. Slower than an A100.
- **A100 (needs Colab Pro).** 40 GB and much faster. This is the card for a full multi-seed evaluation of a large backbone.
- **No GPU (CPU runtime).** Scoring still runs, slowly — see below. This is the one notebook of the three that is genuinely usable without a GPU.

### The free tier will disconnect. Mount Drive before it does.

`Files` — the folder icon in the left margin — then **Mount Drive**, and let Colab run the cell it offers you. Mounting does not stop the disconnect. It gives you one folder, `/content/drive/MyDrive/`, that survives one: everything else in the runtime is thrown away when the session ends, including `/content`, the multi-gigabyte backbone download and anything the panel had written. So mount it before you start something long, and copy what you want to keep into it as it appears — `model_bundle.zip` above all, which is small and is the whole trained model.

### Scoring is much cheaper than training

Prediction is one forward pass per variant with no gradients, so the guide above is stricter than this notebook needs. **A free T4 is enough for every backbone that fits on one**, including the 650M-parameter ones that are painful to *train* there.

Two things still bite. A model trained on one of the three backbones the registry says will not fit a T4 (`SaProt-1.3B`, `ProtT5-XL`, `Ankh-large`) needs the same card here that it needed there — the weights are the same size whether you are training them or not. And a screen of tens of thousands of variants is still a long job: the panel costs it out before it starts, from the size of the backbone and the length of your protein. Read that estimate before you press the button.

**Without a GPU this notebook still works**, on the CPU, at roughly the speed you would expect: fine for a few dozen designs you want to rank before ordering, painful for a library-sized screen.

In [ ]:
#@title **Click the run-button to score variants** { display-mode: "form" }

#@markdown ### Hint
#@markdown - The first run of this cell installs ColabSeqDisplay: **1-2 minutes** on a fresh runtime, with nothing for you to do while it works. Run it again later in the same session and it skips straight to the panel.
#@markdown - **The panel this cell draws below itself is the whole program.** Answer what it asks, press the buttons it offers. There is no code to write and nothing else in this notebook to edit.
#@markdown - **What the run-button is telling you.** The ▶ arrow means nothing is running: click it to start. It spins while the cell installs the package and builds the panel, then goes back to ▶ — that means finished, not broken. The panel stays live after the cell ends, for as long as this runtime does; if it ever stops responding, click ▶ again to rebuild it.
#@markdown ### <font color=red>If the session disconnects</font>
#@markdown - <font color=red>Colab drops long sessions, and the free T4 drops them soonest. Reconnect, run this cell again, and the panel comes back — but the runtime is empty: whatever was under `/content` is gone, whatever you wrote to a mounted Google Drive folder is not. Mount Drive before you start anything long.</font>
#@markdown - <font color=red>Changing the runtime type restarts Python and empties it just the same. Stop this cell first, change the runtime, then run it again.</font>
#@markdown ### Where the code comes from
#@markdown - The field below names what gets installed — **one repository, and it is the whole program**: the fine-tuning engine ships inside it. A **folder path** works as well as a URL — the path of a checkout you uploaded to this runtime — and is installed with `pip install -e`, which keeps its `config/best/` registry and its bundled `examples/` where you can read and edit them.
#@markdown - <font color=red>If this runtime cannot reach that address the cell stops there, quotes what `git` or `pip` said, and names this field as the one to change.</font>
colabsd_repository = "https://github.com/JasonJiangs/ColabSeqDisplay.git"  #@param {type:"string"}

import importlib
import importlib.util
import subprocess
import sys
from pathlib import Path

WORK_ROOT = Path.cwd()


def run_command(command):
    """Run a command, raising with its own output when it fails."""
    parts = [str(part) for part in command]
    finished = subprocess.run(parts, capture_output=True, text=True)
    if finished.returncode != 0:
        raise RuntimeError(
            "This command failed:\n  " + " ".join(parts) + "\n"
            + (finished.stdout or "")[-1500:] + (finished.stderr or "")[-1500:]
        )
    return finished


def checkout(source, name):
    """A local folder as given, or a shallow clone of a git URL beside this notebook."""
    local = Path(source).expanduser()
    if local.is_dir():
        return local.resolve()
    target = WORK_ROOT / name
    if not (target / ".git").is_dir():
        print("cloning " + str(source) + " ...")
        try:
            run_command(["git", "clone", "--depth", "1", source, target])
        except RuntimeError as exc:
            raise RuntimeError(
                str(exc) + "\n\n" + name + " could not be downloaded from " + str(source) + ". Put a "
                "repository this runtime can reach in the field at the top of this form, or the path of a "
                "folder you uploaded to this runtime (for example " + str(target) + ")."
            ) from None
    return target.resolve()


def package_dir(module):
    """The directory an importable package sits in, or None when it is not importable."""
    found = importlib.util.find_spec(module)
    if found is None or not found.origin:
        return None
    return Path(found.origin).resolve().parent


def colabsd_is_complete():
    """True when colabsd is importable *and* its config registry and bundled example came with it.

    Two layouts are both correct: an editable install leaves `config/` and `examples/` beside the
    package, a built wheel carries them inside it. Either answer counts; neither does.
    """
    package = package_dir("colabsd")
    if package is None:
        return False
    return any(
        (root / "config" / "best").is_dir() and (root / "examples").is_dir()
        for root in (package, package.parent)
    )


if not colabsd_is_complete():
    package_root = checkout(colabsd_repository, "ColabSeqDisplay")
    print("installing ColabSeqDisplay from " + str(package_root) + " ...")
    run_command([sys.executable, "-m", "pip", "install", "-q", "-e", package_root])
    importlib.invalidate_caches()
    if str(package_root) not in sys.path:
        sys.path.insert(0, str(package_root))

import colabsd
from colabsd.ui import core, predict_workflow

runtime = core.detect_runtime()
WORK_DIR = WORK_ROOT / "colabsd_work"

if runtime.has_gpu:
    GPU_DESCRIPTION = str(runtime.gpu_name) + "  (" + format(runtime.gpu_memory_gb or 0.0, ".1f") + " GB)"
else:
    GPU_DESCRIPTION = "none — Runtime > Change runtime type > T4 GPU, then run this cell again"

print("colabsd " + colabsd.__version__ + "   from " + str(Path(colabsd.REPO_ROOT)))
print("GPU       " + GPU_DESCRIPTION)
print("files     " + str(WORK_DIR))
print("")

wizard = predict_workflow.launch(work_dir=WORK_DIR, has_gpu=runtime.has_gpu)